<a href="https://colab.research.google.com/github/annnderson/instagram-barbearia/blob/main/Studio_Faria_%E2%80%94_Coleta_Autom%C3%A1tica_Instagram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalar as bibliotecas

In [17]:
!pip install gspread google-auth requests

 Importar as bibliotecas

In [18]:
import requests
import gspread
from google.colab import auth
from google.auth import default
import pandas as pd
from datetime import datetime, timedelta

Configurar o token da API

In [3]:
ACCESS_TOKEN = TOKENBASE_URL = "https://graph.instagram.com"

 Funções de busca

In [19]:
def get_all_posts():
    posts = []
    url = f"{BASE_URL}/me/media"
    params = {
        "fields": "id,caption,timestamp,media_type",
        "access_token": ACCESS_TOKEN
    }
    while url:
        response = requests.get(url, params=params)
        data = response.json()
        if "data" in data:
            posts.extend(data["data"])
            url = data.get("paging", {}).get("next")
            params = {}
        else:
            print("Erro:", data)
            break
    print(f"Total de posts encontrados: {len(posts)}")
    return posts

def get_post_metrics(post_id, media_type):
    url = f"{BASE_URL}/{post_id}/insights"
    params = {
        "metric": "reach,likes,comments,shares,saved",
        "access_token": ACCESS_TOKEN
    }
    response = requests.get(url, params=params)
    data = response.json()
    result = {}
    if "data" in data:
        for item in data["data"]:
            result[item["name"]] = item["values"][0]["value"]
    return result

Autenticar no Google

In [20]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Conectado ao Google!")

✅ Conectado ao Google!


Conectar na planilha

In [21]:
sh = gc.open("Database - Studio Faria")
worksheet = sh.worksheet("Dados dos Posts")
print(f"✅ Planilha: {sh.title} | Aba: {worksheet.title}")

✅ Planilha: Database - Studio Faria | Aba: Dados dos Posts


Buscar e salvar posts novos

In [25]:
# Mapeamento de tipos
tipo_map = {
    "VIDEO": "Reels",
    "IMAGE": "Foto (Feed)",
    "CAROUSEL_ALBUM": "Carrossel",
    "STORY": "Story"
}

dias = {
    "Monday": "segunda-feira", "Tuesday": "terça-feira",
    "Wednesday": "quarta-feira", "Thursday": "quinta-feira",
    "Friday": "sexta-feira", "Saturday": "sábado", "Sunday": "domingo"
}
meses = {
    "January": "janeiro", "February": "fevereiro", "March": "março",
    "April": "abril", "May": "maio", "June": "junho",
    "July": "julho", "August": "agosto", "September": "setembro",
    "October": "outubro", "November": "novembro", "December": "dezembro"
}

# Busca todos os dados existentes
dados_existentes = worksheet.get_all_values()
cabecalho = dados_existentes[0]
linhas_existentes = dados_existentes[1:]

# Pega IDs já existentes na última coluna
col_id = cabecalho.index("ID") if "ID" in cabecalho else -1
ids_existentes = [row[col_id] for row in linhas_existentes if len(row) > col_id and row[col_id]] if col_id >= 0 else []

print(f"IDs já cadastrados: {len(ids_existentes)}")

# Busca posts da API
posts = get_all_posts()
novos = []

for post in posts:
    media_type = post["media_type"]
    post_id = post["id"]

    # Só processa Feed, Reels e Carrossel
    if media_type not in ["IMAGE", "VIDEO", "CAROUSEL_ALBUM"]:
        continue

    # Verifica se já existe pelo ID
    if post_id in ids_existentes:
        print(f"Post {post_id} já existe — pulando")
        continue

    metrics = get_post_metrics(post_id, media_type)

    reach = metrics.get("reach", 0)
    likes = metrics.get("likes", 0)
    comments = metrics.get("comments", 0)
    shares = metrics.get("shares", 0)
    saved = metrics.get("saved", 0)
    interacoes = likes + comments + shares + saved
    engajamento = round((interacoes / reach * 100), 2) if reach > 0 else 0

    timestamp_utc = datetime.strptime(post["timestamp"], "%Y-%m-%dT%H:%M:%S+0000")
    timestamp_br = timestamp_utc - timedelta(hours=3)

    data = timestamp_br.strftime("%d/%m/%Y")
    horario = timestamp_br.strftime("%H:%M")
    dia_semana = dias[timestamp_br.strftime("%A")]
    mes = meses[timestamp_br.strftime("%B")]
    tipo_post = tipo_map.get(media_type, media_type)
    caption = post.get("caption", "")

    novos.append([
        data, tipo_post, "", caption, horario,
        dia_semana, mes, 0, reach, likes, comments,
        saved, shares, 0, interacoes, engajamento, 0, 0, post_id
    ])

if novos:
    for nova_linha in novos:
        try:
            data_nova = datetime.strptime(nova_linha[0], "%d/%m/%Y")
        except:
            continue

        # Encontra a posição certa para inserir
        posicao = len(linhas_existentes) + 2  # padrão: final
        for i, row in enumerate(linhas_existentes):
            try:
                data_existente = datetime.strptime(row[0], "%d/%m/%Y")
                if data_nova <= data_existente:
                    posicao = i + 2  # +2 por causa do cabeçalho
                    break
            except:
                continue

        # Insere na posição certa
        worksheet.insert_row(nova_linha, posicao)
        linhas_existentes.insert(posicao - 2, nova_linha)
        print(f"✅ Inserido na linha {posicao}: {nova_linha[0]} - {nova_linha[1]}")

    print(f"\n✅ Total: {len(novos)} novos posts adicionados!")
else:
    print("ℹ️ Nenhum post novo encontrado.")

IDs já cadastrados: 9
Total de posts encontrados: 9
Post 17851603830624527 já existe — pulando
Post 18100569217892576 já existe — pulando
Post 18339124942240663 já existe — pulando
Post 18060472151247569 já existe — pulando
Post 18315646393123506 já existe — pulando
Post 18026353631001021 já existe — pulando
Post 18026160262653379 já existe — pulando
Post 18235301791200374 já existe — pulando
Post 17869625309866139 já existe — pulando
ℹ️ Nenhum post novo encontrado.
